Power Outage Final Project
**Name**: Patrick Wu

**Website Link**: https://exitcy.github.io/power-outage-analysis-ml/

In [1]:
# Import libraries used across all steps: pandas/numpy for data, Path for asset paths, Plotly for plots.
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'


## Step 1: Introduction

### Understanding the data

This project uses the **U.S. major power outage** dataset (`outage.xlsx`): one row per major outage in the continental U.S. from **January 2000 through July 2016**, with **1,535** event records and **57** variables after loading with `skiprows=5`.

Each row describes **when and where** an outage happened, **what caused** it, **how long** it lasted, and **how many customers** were affected, together with **climate**, **electricity price/sales**, and **state economic/demographic** context for the affected area. That mix supports both exploratory questions (distributions, geography, causes) and predictive work (regression on duration or classification on cause).

### Questions that interest me

1. **Where and when** do major outages cluster—by state, climate region, month, or year?
2. **What cause categories** are most common, and do causes with different origins (e.g., severe weather vs. intentional attack) show different **severity** (customers affected, demand loss, duration)?
3. **How predictable is outage duration** from early context (location, season, climate anomaly, grid scale, cause) that planners might know soon after an event starts?
4. **Is duration prediction equally accurate** for high-impact areas (many total customers) vs. lower-impact areas—an equity question for resource allocation?

These may change as the project progresses; Steps 4–8 below focus especially on **duration** and the factors tied to it.

### Question chosen for this project

**How long do major power outages last, and what factors—especially outage cause—help explain or predict that duration?**

I chose this question because it connects directly to grid **resilience and public infrastructure planning**: longer outages mean longer service disruption. My Step 4 permutation test compares **severe weather** vs. **intentional attack** average durations. Steps 5–7 frame **regression** with target `OUTAGE.DURATION`, using location, climate, seasonality, customer scale, and (in the final model) cause category—features justified by the strong cause–duration relationship found in Step 4. Step 8 then asks whether the **duration model** behaves similarly for high- vs. low–customer-scale outages. If this question narrows further, it will stay centered on **duration** rather than switching to a wholly different outcome.

**Why this dataset:** The power outage dataset lets me apply course methods to data with real-world stakes—understanding and anticipating outage length supports crew deployment and resilience planning. The same table also supports other tasks (e.g., classifying cause or predicting customers affected), but **duration** is the thread that best matches the analysis already completed in this notebook.

In [2]:
# Step 1: Dataset size and columns relevant to the chosen question

project_question = (
    'How long do major power outages last, and what factors—especially outage cause—'
    'help explain or predict that duration?'
)
print('Project question:')
print(project_question)
print()

df_step1 = pd.read_excel('outage.xlsx', skiprows=5)
if df_step1.iloc[0].isnull().all() or 'variables' in str(df_step1.iloc[0].values).lower():
    df_step1 = df_step1.iloc[1:].reset_index(drop=True)

print(f'Number of rows in the dataset: {len(df_step1)}')
print(f'Number of columns: {df_step1.shape[1]}')

relevant_columns = {
    'OUTAGE.DURATION': (
        'Length of the outage in minutes (target for regression in Steps 5–7; '
        'compared across cause groups in Step 4).'
    ),
    'CAUSE.CATEGORY': (
        'Broad cause label (e.g., severe weather, intentional attack, equipment failure). '
        'Used in Step 4 hypothesis test and as a predictor in the final duration model.'
    ),
    'U.S._STATE': (
        'U.S. state where the outage occurred; geographic context for EDA and models.'
    ),
    'CLIMATE.REGION': (
        'NOAA-style climate region for the affected area; captures regional weather patterns.'
    ),
    'MONTH': (
        'Calendar month of outage start (1–12); encodes seasonality in baseline and final models.'
    ),
    'ANOMALY.LEVEL': (
        'Climate anomaly level associated with the event; numeric context for outage conditions.'
    ),
    'TOTAL.CUSTOMERS': (
        'Total electricity customers in the affected area; proxy for grid scale and Step 8 impact groups.'
    ),
    'CUSTOMERS.AFFECTED': (
        'Number of customers who lost power; severity measure with substantial missingness (Step 3).'
    ),
    'YEAR': (
        'Year of the outage (2000–2016); useful for temporal trends in EDA.'
    ),
    'OUTAGE.START.DATE': (
        'Date the outage began (combined with start time in Step 2 cleaning).'
    ),
    'OUTAGE.START.TIME': (
        'Time the outage began; paired with start date to build a timestamp.'
    ),
}

relevant_df = pd.DataFrame(
    {'description': list(relevant_columns.values())},
    index=list(relevant_columns.keys()),
)
relevant_df.index.name = 'column'
display(relevant_df)

Project question:
How long do major power outages last, and what factors—especially outage cause—help explain or predict that duration?

Number of rows in the dataset: 1535
Number of columns: 57


,description
column,
OUTAGE.DURATION,Length of the outage in minutes (target for re...
CAUSE.CATEGORY,"Broad cause label (e.g., severe weather, inten..."
U.S._STATE,U.S. state where the outage occurred; geograph...
CLIMATE.REGION,NOAA-style climate region for the affected are...
MONTH,Calendar month of outage start (1–12); encodes...
ANOMALY.LEVEL,Climate anomaly level associated with the even...
TOTAL.CUSTOMERS,Total electricity customers in the affected ar...
CUSTOMERS.AFFECTED,Number of customers who lost power; severity m...
YEAR,Year of the outage (2000–2016); useful for tem...


## Step 2: Data Cleaning and Exploratory Data Analysis

In [3]:
# Step 2: Load outage data, clean duration/cause fields, and produce EDA plots plus summary tables.
# Each plot is saved to assets/ where noted and is labeled so it stands alone without reading other cells.

ASSETS_DIR = Path('assets')
ASSETS_DIR.mkdir(exist_ok=True)


def clean_duration_column(frame):
    """Strip text suffixes from duration and coerce to numeric minutes."""
    out = frame.copy()
    out['OUTAGE.DURATION'] = (
        out['OUTAGE.DURATION']
        .astype(str)
        .str.replace('mins', '', case=False)
        .str.strip()
    )
    out['OUTAGE.DURATION'] = pd.to_numeric(out['OUTAGE.DURATION'], errors='coerce')
    return out


# Read Excel (first 5 rows are metadata); drop the variable-description row if present.
df = pd.read_excel('outage.xlsx', skiprows=5)
if df.iloc[0].isnull().all() or 'variables' in str(df.iloc[0].values).lower():
    df = df.iloc[1:].reset_index(drop=True)

# Standardize duration and cause labels for grouping in later steps.
df = clean_duration_column(df)
df['CAUSE.CATEGORY_CLEAN'] = (
    df['CAUSE.CATEGORY'].astype(str).str.strip().str.lower()
)

bivariate_df = df.dropna(subset=['CAUSE.CATEGORY', 'OUTAGE.DURATION']).copy()

print('Cleaned DataFrame head (relevant columns):')
display(
    bivariate_df[
        ['OUTAGE.DURATION', 'CAUSE.CATEGORY', 'U.S._STATE', 'CLIMATE.REGION', 'MONTH', 'TOTAL.CUSTOMERS']
    ].head()
)

# Univariate plot 1: how often each cause category appears.
print('\n### Univariate: outage cause counts')
plot_df = df.dropna(subset=['CAUSE.CATEGORY'])
fig_cause = px.histogram(
    plot_df,
    x='CAUSE.CATEGORY',
    title='Distribution of Major Power Outage Causes in the U.S. (2000–2016)',
    labels={'CAUSE.CATEGORY': 'Cause Category', 'count': 'Number of Outages'},
    color='CAUSE.CATEGORY',
    color_discrete_sequence=px.colors.qualitative.Safe,
)
fig_cause.update_layout(
    xaxis_title='Outage Cause Category',
    yaxis_title='Count of Outages',
    showlegend=True,
    legend_title='Cause Category',
    plot_bgcolor='white',
    title_font_size=18,
)
fig_cause.update_xaxes(showline=True, linewidth=1, linecolor='lightgray', tickangle=45)
fig_cause.update_yaxes(showline=True, linewidth=1, linecolor='lightgray', gridcolor='whitesmoke')
fig_cause.write_html(ASSETS_DIR / 'univariate_cause_histogram.html', include_plotlyjs='cdn')
fig_cause.show()

# Univariate plot 2: distribution of outage length in minutes.
print('\n### Univariate: outage duration distribution')
duration_df = df.dropna(subset=['OUTAGE.DURATION']).copy()
fig_duration = px.histogram(
    duration_df,
    x='OUTAGE.DURATION',
    nbins=50,
    title='Distribution of Outage Duration (minutes)',
    labels={'OUTAGE.DURATION': 'Duration (minutes)', 'count': 'Number of Outages'},
)
fig_duration.update_layout(
    xaxis_title='Outage Duration (minutes)',
    yaxis_title='Number of Outages',
    plot_bgcolor='white',
    title_font_size=18,
)
fig_duration.update_xaxes(showline=True, linewidth=1, linecolor='lightgray')
fig_duration.update_yaxes(showline=True, linewidth=1, linecolor='lightgray', gridcolor='whitesmoke')
fig_duration.write_html(ASSETS_DIR / 'univariate_duration_histogram.html', include_plotlyjs='cdn')
fig_duration.show()

# Bivariate plot 1: compare duration spread across cause categories.
print('\n### Bivariate: duration by cause category')
fig_duration_cause = px.box(
    bivariate_df,
    x='CAUSE.CATEGORY',
    y='OUTAGE.DURATION',
    title='Outage Duration by Cause Category',
    labels={'CAUSE.CATEGORY': 'Cause Category', 'OUTAGE.DURATION': 'Duration (minutes)'},
    color='CAUSE.CATEGORY',
    color_discrete_sequence=px.colors.qualitative.Safe,
)
fig_duration_cause.update_layout(
    xaxis_title='Outage Cause Category',
    yaxis_title='Outage Duration (minutes)',
    showlegend=True,
    legend_title='Cause Category',
    plot_bgcolor='white',
    title_font_size=18,
)
fig_duration_cause.update_xaxes(showline=True, linewidth=1, linecolor='lightgray', tickangle=45)
fig_duration_cause.update_yaxes(showline=True, linewidth=1, linecolor='lightgray', gridcolor='whitesmoke')
fig_duration_cause.write_html(ASSETS_DIR / 'bivariate_duration_by_cause.html', include_plotlyjs='cdn')
fig_duration_cause.show()

# Bivariate plot 2: relationship between grid size and outage length, colored by cause.
print('\n### Bivariate: duration vs total customers (log scale)')
scatter_df = bivariate_df.dropna(subset=['TOTAL.CUSTOMERS']).copy()
fig_customers = px.scatter(
    scatter_df,
    x='TOTAL.CUSTOMERS',
    y='OUTAGE.DURATION',
    color='CAUSE.CATEGORY',
    title='Outage Duration vs Grid Size (Total Customers)',
    labels={
        'TOTAL.CUSTOMERS': 'Total Customers in Affected Area',
        'OUTAGE.DURATION': 'Duration (minutes)',
        'CAUSE.CATEGORY': 'Cause Category',
    },
    log_x=True,
    opacity=0.6,
)
fig_customers.update_layout(
    xaxis_title='Total Customers in Affected Area (log scale)',
    yaxis_title='Outage Duration (minutes)',
    legend_title='Cause Category',
    plot_bgcolor='white',
    title_font_size=18,
)
fig_customers.write_html(ASSETS_DIR / 'bivariate_duration_vs_customers.html', include_plotlyjs='cdn')
fig_customers.show()

# Summary table: mean/median duration and count for each cause.
agg_table = (
    bivariate_df.groupby('CAUSE.CATEGORY')['OUTAGE.DURATION']
    .agg(count='count', mean_minutes='mean', median_minutes='median')
    .reset_index()
    .sort_values('count', ascending=False)
)
agg_table['mean_minutes'] = agg_table['mean_minutes'].round(2)
agg_table['median_minutes'] = agg_table['median_minutes'].round(2)

print('\n### Interesting aggregates: duration summary by cause category')
display(agg_table)


Cleaned DataFrame head (relevant columns):


,OUTAGE.DURATION,CAUSE.CATEGORY,U.S._STATE,CLIMATE.REGION,MONTH,TOTAL.CUSTOMERS
1,3060.0,severe weather,Minnesota,East North Central,7.0,2595696.0
2,1.0,intentional attack,Minnesota,East North Central,5.0,2640737.0
3,3000.0,severe weather,Minnesota,East North Central,10.0,2586905.0
4,2550.0,severe weather,Minnesota,East North Central,6.0,2606813.0
5,1740.0,severe weather,Minnesota,East North Central,7.0,2673531.0



### Univariate: outage cause counts



### Univariate: outage duration distribution



### Bivariate: duration by cause category



### Bivariate: duration vs total customers (log scale)



### Interesting aggregates: duration summary by cause category


,CAUSE.CATEGORY,count,mean_minutes,median_minutes
5,severe weather,744,3883.99,2460.0
2,intentional attack,403,429.98,56.0
6,system operability disruption,123,728.87,215.0
4,public appeal,69,1468.45,455.0
0,equipment failure,55,1816.91,221.0
3,islanding,44,200.55,77.5
1,fuel supply emergency,38,13484.03,3960.0


## Step 3: Assessment of Missingness

### What this step is about

In Steps 1–2 we cleaned the data and explored outage duration and cause. Before building prediction models, we need to understand **why `CUSTOMERS.AFFECTED` is often missing** (~28.9% of rows) and whether that missingness is related to other variables we can see in the table.

This step has two parts:

1. **NMAR reasoning** — Use the data generating process (utility OE-417 reporting) to argue whether missing customer-impact counts could be **Not Missing At Random**, and what extra data would help explain them.
2. **Missingness dependency tests** — Use **permutation tests** to check whether the *pattern* of missingness depends on observed columns. That tells us if the data look more like **MCAR**, **MAR**, or still plausibly **NMAR** on top of MAR.

We do **not** impute or drop rows here; we only characterize missingness mechanisms for the report and for interpreting later models (where we avoid using `CUSTOMERS.AFFECTED` as a predictor because it is often missing at prediction time).

---

### NMAR Analysis

We focus on **`CUSTOMERS.AFFECTED`**. We believe it is plausibly **NMAR**: whether the count is recorded may depend on unobserved reporting workflow (survey timing, security redaction for attacks, internal thresholds) or on the true impact itself—not only on columns in this spreadsheet.

**Important:** Permutation tests below can show dependency on **observed** columns (MAR vs. MCAR), but they **cannot prove or disprove NMAR** by themselves. To move toward MAR we would want extra fields such as OE-417 submission dates, regulatory completeness flags, and reporting-threshold rules by utility and year.

---

### Missingness dependency tests (code below)

**Question:** Is missingness in `CUSTOMERS.AFFECTED` independent of other columns?

**Method:** For each comparison column, we shuffle the missingness indicator 2,000 times (holding the comparison column fixed), build a null distribution of a test statistic, and compute a p-value. Significance level **α = 0.05**.

| Test | Column | Statistic | Expectation |
|------|--------|-----------|-------------|
| 1 (dependent) | `CAUSE.CATEGORY` | Variance of group-wise missingness rates (per cause, including `__MISSING__` cause labels) | Reject independence — missingness varies by cause |
| 2 (control) | `TOTAL.CUSTOMERS` | Absolute difference in mean grid size when impact is missing vs observed | Fail to reject — grid scale does not drive missingness |

**Plots exported to `assets/`:**

- Cause distribution when impact is **missing vs observed** (Lecture 8 style)
- Permutation null for the **cause** test
- Box plot and permutation null for the **total customers** control test

**Takeaway:** If Test 1 is significant and Test 2 is not, missingness depends on at least one observed column → pattern is more consistent with **MAR** than **MCAR**, while NMAR from the reporting process may still apply.

In [4]:
# Step 3: Test whether missingness in CUSTOMERS.AFFECTED depends on other observed columns.
# Permutation tests compare an observed statistic to a null built by shuffling missingness labels.

rng = np.random.default_rng(80)

nmar_col = 'CUSTOMERS.AFFECTED'
miss = df[nmar_col].isna()

print(f"Missingness rate for {nmar_col}: {miss.mean():.3f}")


def permutation_pvalue_for_categorical_missingness(data, target_missing, group_col, reps=2000):
    """Test whether missingness depends on a categorical column (statistic: variance of group missing rates)."""
    x = data[group_col].fillna('__MISSING__').astype(str).to_numpy()
    y = target_missing.astype(int).to_numpy()

    obs_tab = pd.crosstab(x, y, normalize='index')
    obs_stat = obs_tab.get(1, pd.Series(0.0, index=obs_tab.index)).dropna().var()

    sim_stats = np.empty(reps)
    for i in range(reps):
        y_perm = rng.permutation(y)
        perm_tab = pd.crosstab(x, y_perm, normalize='index')
        sim_stats[i] = perm_tab.get(1, pd.Series(0.0, index=perm_tab.index)).dropna().var()

    p_val = (np.sum(sim_stats >= obs_stat) + 1) / (reps + 1)
    return obs_stat, p_val, sim_stats


def permutation_pvalue_for_numeric_missingness(data, target_missing, numeric_col, reps=2000):
    """Test whether missingness depends on a numeric column (statistic: |mean difference| by missing group)."""
    temp = data[[numeric_col]].copy()
    temp['is_missing'] = target_missing
    temp = temp.dropna()

    x = temp[numeric_col].astype(float).to_numpy()
    y = temp['is_missing'].astype(int).to_numpy()

    obs_stat = abs(x[y == 1].mean() - x[y == 0].mean())

    sim_stats = np.empty(reps)
    for i in range(reps):
        y_perm = rng.permutation(y)
        sim_stats[i] = abs(x[y_perm == 1].mean() - x[y_perm == 0].mean())

    p_val = (np.sum(sim_stats >= obs_stat) + 1) / (reps + 1)
    return obs_stat, p_val, sim_stats


# CAUSE.CATEGORY should show dependence; TOTAL.CUSTOMERS serves as a null-style control.
dep_col = 'CAUSE.CATEGORY'
dep_stat, dep_p, sim_stats = permutation_pvalue_for_categorical_missingness(df, miss, dep_col)

indep_col = 'TOTAL.CUSTOMERS'
indep_stat, indep_p, indep_sim = permutation_pvalue_for_numeric_missingness(df, miss, indep_col)

results_step3 = pd.DataFrame({
    'column_tested': [dep_col, indep_col],
    'test_type': ['categorical missingness-permutation', 'numeric missingness-permutation'],
    'statistic': [dep_stat, indep_stat],
    'p_value': [dep_p, indep_p]
})

print('\nPermutation test results for missingness dependency:')
display(results_step3)

alpha = 0.05
print('\nInterpretation at alpha = 0.05:')
if dep_p < alpha:
    print(f"- Missingness of {nmar_col} appears to depend on {dep_col} (reject independence).")
else:
    print(f"- No strong evidence that missingness of {nmar_col} depends on {dep_col}.")

if indep_p < alpha:
    print(f"- Missingness of {nmar_col} appears to depend on {indep_col}.")
else:
    print(f"- Missingness of {nmar_col} does not appear to depend on {indep_col} (fail to reject independence).")

print("\nConclusion: Because missingness depends on observed data (at least one other column),")
print("this pattern is more consistent with MAR than MCAR. This does NOT prove NMAR by itself.")

# Null distribution for the dependent case; red line marks the observed statistic.
fig_miss = px.histogram(
    sim_stats,
    nbins=40,
    title='Permutation Null: Missingness of CUSTOMERS.AFFECTED vs CAUSE.CATEGORY',
    labels={'value': 'Variance of group missingness rates', 'count': 'Frequency'},
)
fig_miss.add_vline(
    x=dep_stat,
    line_dash='dash',
    line_color='red',
    annotation_text='Observed statistic',
)
fig_miss.update_layout(
    xaxis_title='Variance of group missingness rates (permuted)',
    yaxis_title='Frequency',
    plot_bgcolor='white',
    title_font_size=16,
)
fig_miss.write_html(ASSETS_DIR / 'missingness_permutation_null.html', include_plotlyjs='cdn')
fig_miss.show()

# Lecture 8 style: distribution of CAUSE.CATEGORY when customer impact is missing vs observed
cause_plot_df = df.dropna(subset=['CAUSE.CATEGORY']).copy()
cause_plot_df['impact_status'] = np.where(
    cause_plot_df['CUSTOMERS.AFFECTED'].isna(),
    'CUSTOMERS.AFFECTED missing',
    'CUSTOMERS.AFFECTED observed',
)
cause_counts = (
    cause_plot_df.groupby(['impact_status', 'CAUSE.CATEGORY'], observed=True)
    .size()
    .reset_index(name='count')
)
fig_cause_miss = px.bar(
    cause_counts,
    x='CAUSE.CATEGORY',
    y='count',
    color='impact_status',
    barmode='group',
    title='Distribution of Outage Cause When Customer Impact Is Missing vs Observed',
    labels={
        'CAUSE.CATEGORY': 'Cause Category',
        'count': 'Number of Outages',
        'impact_status': 'Customer impact reporting',
    },
)
fig_cause_miss.update_layout(plot_bgcolor='white', title_font_size=16, xaxis_tickangle=-45)
fig_cause_miss.write_html(ASSETS_DIR / 'missingness_cause_by_impact_status.html', include_plotlyjs='cdn')
fig_cause_miss.show()

# TOTAL.CUSTOMERS: box plot and permutation null (control test)
tc_plot_df = df.dropna(subset=['TOTAL.CUSTOMERS']).copy()
tc_plot_df['impact_status'] = np.where(
    tc_plot_df['CUSTOMERS.AFFECTED'].isna(),
    'CUSTOMERS.AFFECTED missing',
    'CUSTOMERS.AFFECTED observed',
)
fig_tc_miss = px.box(
    tc_plot_df,
    x='impact_status',
    y='TOTAL.CUSTOMERS',
    color='impact_status',
    title='Total Customers When Customer Impact Is Missing vs Observed',
    labels={'TOTAL.CUSTOMERS': 'Total customers', 'impact_status': 'Customer impact reporting'},
)
fig_tc_miss.update_layout(plot_bgcolor='white', title_font_size=16, showlegend=False)
fig_tc_miss.write_html(ASSETS_DIR / 'missingness_total_customers_by_impact_status.html', include_plotlyjs='cdn')
fig_tc_miss.show()

fig_indep = px.histogram(
    indep_sim,
    nbins=40,
    title='Permutation Null: Missingness of CUSTOMERS.AFFECTED vs TOTAL.CUSTOMERS',
    labels={'value': 'Absolute difference in mean TOTAL.CUSTOMERS', 'count': 'Frequency'},
)
fig_indep.add_vline(x=indep_stat, line_dash='dash', line_color='red', annotation_text='Observed statistic')
fig_indep.update_layout(
    xaxis_title='Absolute difference in mean TOTAL.CUSTOMERS (permuted)',
    yaxis_title='Frequency',
    plot_bgcolor='white',
    title_font_size=16,
)
fig_indep.write_html(ASSETS_DIR / 'missingness_total_customers_permutation_null.html', include_plotlyjs='cdn')
fig_indep.show()


Missingness rate for CUSTOMERS.AFFECTED: 0.289

Permutation test results for missingness dependency:


,column_tested,test_type,statistic,p_value
0,CAUSE.CATEGORY,categorical missingness-permutation,0.098105,0.000500
1,TOTAL.CUSTOMERS,numeric missingness-permutation,59563.635640,0.807096



Interpretation at alpha = 0.05:
- Missingness of CUSTOMERS.AFFECTED appears to depend on CAUSE.CATEGORY (reject independence).
- Missingness of CUSTOMERS.AFFECTED does not appear to depend on TOTAL.CUSTOMERS (fail to reject independence).

Conclusion: Because missingness depends on observed data (at least one other column),
this pattern is more consistent with MAR than MCAR. This does NOT prove NMAR by itself.


## Step 4: Hypothesis Testing

We test whether outages caused by **severe weather** have a different average duration than those caused by **intentional attacks**—the two most common cause categories and the pair with the largest apparent duration gap in our EDA.

- **Null Hypothesis (H₀):** The duration of power outages caused by severe weather and those caused by intentional attacks come from the same underlying distribution (equal population means).
- **Alternative Hypothesis (Hₐ):** Power outages caused by severe weather have a different average duration than power outages caused by intentional attacks.

**Test statistic:** Absolute difference in sample means, |x̄_severe − x̄_attack|.

**Significance level:** α = 0.05.

**Method:** Permutation test with 3,000 repetitions, shuffling duration labels between the two groups while holding group sizes fixed.

In [5]:
# Step 4: Two-sample permutation test on mean outage duration (severe weather vs intentional attack).

rng = np.random.default_rng(80)

# Reuse cleaned duration and normalized cause labels from Step 2.
analysis = df[['CAUSE.CATEGORY', 'OUTAGE.DURATION']].copy()
analysis = clean_duration_column(analysis)
analysis['CAUSE.CATEGORY_CLEAN'] = analysis['CAUSE.CATEGORY'].astype(str).str.strip().str.lower()
analysis = analysis.dropna(subset=['OUTAGE.DURATION'])

severe = analysis.loc[analysis['CAUSE.CATEGORY_CLEAN'] == 'severe weather', 'OUTAGE.DURATION'].to_numpy()
attacks = analysis.loc[analysis['CAUSE.CATEGORY_CLEAN'] == 'intentional attack', 'OUTAGE.DURATION'].to_numpy()

if len(severe) == 0 or len(attacks) == 0:
    raise ValueError(
        f"Data extraction failed! Found n={len(severe)} for Severe Weather and n={len(attacks)} for Intentional Attacks. "
        f"Please check your raw unique values using: print(df['CAUSE.CATEGORY'].unique())"
    )

# H0: same underlying duration distribution; HA: different average duration (two-sided via |mean diff|).
observed_stat = abs(severe.mean() - attacks.mean())

combined = np.concatenate([severe, attacks])
n_severe = len(severe)
reps = 3000
simulated_stats = np.empty(reps)

for i in range(reps):
    perm = rng.permutation(combined)
    simulated_stats[i] = abs(perm[:n_severe].mean() - perm[n_severe:].mean())

p_value = (np.sum(simulated_stats >= observed_stat) + 1) / (reps + 1)

print('Hypothesis test: OUTAGE.DURATION by Severe Weather vs Intentional Attacks')
print(f'n(Severe Weather) = {len(severe)}, n(Intentional Attacks) = {len(attacks)}')
print(f'Mean severe-weather duration: {severe.mean():.2f} minutes')
print(f'Mean intentional attack duration: {attacks.mean():.2f} minutes')
print(f'Observed statistic |mean diff|: {observed_stat:.2f}')
print(f'Permutation p-value: {p_value:.5f}')

alpha = 0.05
if p_value < alpha:
    print('Conclusion: Reject H0. Outage duration differs significantly between Severe Weather and Intentional Attacks.')
else:
    print('Conclusion: Fail to reject H0. No statistically significant difference in outage duration.')

fig_hyp = px.histogram(
    simulated_stats,
    nbins=40,
    title='Permutation Null: |Mean Duration Difference| (Severe Weather vs Intentional Attack)',
    labels={'value': '|Difference in sample means| (minutes)', 'count': 'Frequency'},
)
fig_hyp.add_vline(
    x=observed_stat,
    line_dash='dash',
    line_color='red',
    annotation_text='Observed statistic',
)
fig_hyp.update_layout(
    xaxis_title='|Difference in sample means| (minutes)',
    yaxis_title='Frequency',
    plot_bgcolor='white',
    title_font_size=16,
)
fig_hyp.write_html(ASSETS_DIR / 'hypothesis_permutation_null.html', include_plotlyjs='cdn')
fig_hyp.show()


Hypothesis test: OUTAGE.DURATION by Severe Weather vs Intentional Attacks
n(Severe Weather) = 744, n(Intentional Attacks) = 403
Mean severe-weather duration: 3883.99 minutes
Mean intentional attack duration: 429.98 minutes
Observed statistic |mean diff|: 3454.01
Permutation p-value: 0.00033
Conclusion: Reject H0. Outage duration differs significantly between Severe Weather and Intentional Attacks.


## Step 5: Framing a Prediction Problem

**Prediction problem:** Given contextual information available early in a major outage event, predict how long the outage will last (in minutes).

**Problem type:** **Regression** — the response variable is continuous.

**Response variable:** `OUTAGE.DURATION` (minutes). This is the direct quantitative answer to our research question; cause category is strongly associated with duration in Step 4.

**Evaluation metric:** **Root Mean Squared Error (RMSE)** on held-out test data (20% split, `random_state=80`). RMSE penalizes large prediction errors more heavily than mean absolute error, which matters when underestimating a multi-day outage is costlier than a small timing error. We report R² as a secondary metric to describe variance explained.

**Time-of-prediction justification (avoiding data leakage):**

| Feature | Available at prediction time? | Rationale |
| --- | --- | --- |
| `MONTH` | Yes | Outage start month is recorded at event onset. |
| `U.S._STATE` | Yes | Location is known immediately. |
| `CLIMATE.REGION` | Yes | Derived from state/region mapping in the dataset. |
| `ANOMALY.LEVEL` | Yes | Climate context for the event period is observable at start. |
| `TOTAL.CUSTOMERS` | Yes | Grid scale for the affected area is a fixed infrastructure attribute. |
| `CAUSE.CATEGORY` | Yes (final model) | Initial cause classification is assigned early in the reporting process. |
| `OUTAGE.DURATION` | **No** | This is the target—we cannot use the answer to predict itself. |
| `OUTAGE.RESTORATION.DATE/TIME` | **No** | Restoration timestamps are only known after the outage ends. |
| `CUSTOMERS.AFFECTED` | **No** | Often missing or updated after initial report. |
| Post-outcome economic fields | **No** | Price, sales, and GSP figures describe the billing period, not early outage conditions. |

The baseline model (Step 6) excludes `CAUSE.CATEGORY` to establish a lower bound using only location, season, climate, and grid scale. The final model (Step 7) adds cause category because it is classified early and our hypothesis test showed it is strongly predictive.

In [6]:
# Step 5: Define the supervised learning task—predict continuous outage duration (minutes).
# We evaluate with RMSE (primary) and R² (secondary) on a held-out test split in Steps 6–7.

prediction_target = 'OUTAGE.DURATION'
print('Step 5 prediction problem defined:')
print('- Task: Regression')
print('- Target: OUTAGE.DURATION (Continuous minutes)')
print('- Main evaluation metric: Root Mean Squared Error (RMSE) on held-out test data')
print('- Secondary metric: R^2 Score')


Step 5 prediction problem defined:
- Task: Regression
- Target: OUTAGE.DURATION (Continuous minutes)
- Main evaluation metric: Root Mean Squared Error (RMSE) on held-out test data
- Secondary metric: R^2 Score


## Step 6: Baseline Model

**Model:** `LinearRegression` in a single scikit-learn `Pipeline` (preprocessing + estimator).

**Features (5 total):** `MONTH` and `TOTAL.CUSTOMERS` (quantitative), `ANOMALY.LEVEL` (ordinal), `CLIMATE.REGION` and `U.S._STATE` (nominal). Numeric features are median-imputed and scaled; categoricals are mode-imputed and one-hot encoded.

**Train/test split:** 80/20 hold-out, `random_state=80`.

**Expectation:** This baseline should perform poorly without `CAUSE.CATEGORY`. Step 4 showed cause drives large duration differences that a linear model on location and climate alone cannot capture. We compare test RMSE to a mean-prediction benchmark to confirm whether the model adds any value.

In [7]:
# Step 6: Baseline linear regression on location, climate, seasonality, and grid-scale features.
# A mean-prediction benchmark shows whether the model beats always guessing the training average.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df_baseline = df.copy()

feature_cols = [
    'MONTH',
    'ANOMALY.LEVEL',
    'TOTAL.CUSTOMERS',
    'CLIMATE.REGION',
    'U.S._STATE',
]

model_df = df_baseline[feature_cols + ['OUTAGE.DURATION']].dropna(subset=['OUTAGE.DURATION']).copy()
model_df = clean_duration_column(model_df)

X = model_df[feature_cols]
y = model_df['OUTAGE.DURATION'].astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=80,
)

numeric_features = ['MONTH', 'ANOMALY.LEVEL', 'TOTAL.CUSTOMERS']
categorical_features = ['CLIMATE.REGION', 'U.S._STATE']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

baseline_model = Pipeline([
    ('preprocess', preprocessor),
    ('reg', LinearRegression()),
])

baseline_model.fit(X_train, y_train)
y_pred = baseline_model.predict(X_test)

baseline_rmse = root_mean_squared_error(y_test, y_pred)
baseline_r2 = r2_score(y_test, y_pred)

mean_duration_value = y_train.mean()
mean_pred = np.full(shape=len(y_test), fill_value=mean_duration_value)
mean_rmse = root_mean_squared_error(y_test, mean_pred)
mean_r2 = r2_score(y_test, mean_pred)

print('Baseline Model: LinearRegression Pipeline')
print(f'Test RMSE: {baseline_rmse:.2f} minutes')
print(f'Test R^2 score: {baseline_r2:.4f}')
print('\nMean-Prediction Reference (always predicting the training mean duration):')
print(f'Reference RMSE: {mean_rmse:.2f} minutes')
print(f'Reference R^2 score: {mean_r2:.4f}')


Baseline Model: LinearRegression Pipeline
Test RMSE: 5179.97 minutes
Test R^2 score: 0.0048

Mean-Prediction Reference (always predicting the training mean duration):
Reference RMSE: 5194.47 minutes
Reference R^2 score: -0.0008


## Step 7: Final Model



In [8]:
# Step 7: Final Random Forest model with richer features (including cause) and hyperparameter tuning.
# Recomputes the Step 6 baseline on the same split for a fair RMSE comparison.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    QuantileTransformer,
    FunctionTransformer,
)
from sklearn.ensemble import RandomForestRegressor

# Reload raw data if this cell is run without Step 2 in the same kernel.
if 'df' not in globals():
    df = pd.read_excel('outage.xlsx', skiprows=5)
    if df.iloc[0].isnull().all() or 'variables' in str(df.iloc[0].values).lower():
        df = df.iloc[1:].reset_index(drop=True)

feature_cols = [
    'MONTH',
    'ANOMALY.LEVEL',
    'TOTAL.CUSTOMERS',
    'CLIMATE.REGION',
    'U.S._STATE',
]


def _clean_duration_column(frame):
    out = frame.copy()
    out['OUTAGE.DURATION'] = (
        out['OUTAGE.DURATION']
        .astype(str)
        .str.replace('mins', '', case=False)
        .str.strip()
    )
    out['OUTAGE.DURATION'] = pd.to_numeric(out['OUTAGE.DURATION'], errors='coerce')
    return out.dropna(subset=['OUTAGE.DURATION'])


def log1p_customers(X):
    """Right-skewed customer counts -> log scale for tree splits."""
    arr = np.asarray(X, dtype=float)
    return np.log1p(np.maximum(arr, 0)).reshape(-1, 1)


def month_seasonality(X):
    """Cyclic month feature (seasonality is not linear in 1..12)."""
    month = np.asarray(X, dtype=float).ravel()
    return np.sin(2 * np.pi * month / 12).reshape(-1, 1)


def anomaly_as_numeric(X):
    return pd.to_numeric(np.asarray(X).ravel(), errors='coerce').reshape(-1, 1)


# Re-fit the Step 6 linear baseline on the same 80/20 split for comparison.
baseline_df = _clean_duration_column(
    df[feature_cols + ['OUTAGE.DURATION']].dropna(subset=['OUTAGE.DURATION'])
)
X_base = baseline_df[feature_cols]
y_base = baseline_df['OUTAGE.DURATION'].astype(float)
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_base, y_base, test_size=0.2, random_state=80
)
numeric_features = ['MONTH', 'ANOMALY.LEVEL', 'TOTAL.CUSTOMERS']
categorical_features = ['CLIMATE.REGION', 'U.S._STATE']
baseline_preprocessor = ColumnTransformer([
    (
        'num',
        Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]),
        numeric_features,
    ),
    (
        'cat',
        Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]),
        categorical_features,
    ),
])
baseline_model = Pipeline([
    ('preprocess', baseline_preprocessor),
    ('reg', LinearRegression()),
])
baseline_model.fit(X_train_base, y_train_base)
baseline_rmse = root_mean_squared_error(
    y_test_base, baseline_model.predict(X_test_base)
)

final_feature_cols = feature_cols + ['CAUSE.CATEGORY']
# Final model adds CAUSE.CATEGORY and engineered numeric transforms before a tuned random forest.
final_model_df = _clean_duration_column(
    df[final_feature_cols + ['OUTAGE.DURATION']].dropna(subset=['OUTAGE.DURATION'])
)

X_final = final_model_df[final_feature_cols]
y_final = final_model_df['OUTAGE.DURATION'].astype(float)

# Hold out the same 20% test rows as the baseline (random_state=80).
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X_final, y_final, test_size=0.2, random_state=80
)

final_categorical = ['CLIMATE.REGION', 'U.S._STATE', 'CAUSE.CATEGORY']

numeric_base_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
log_customers_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(log1p_customers, validate=False)),
])
quantile_anomaly_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('to_num', FunctionTransformer(anomaly_as_numeric, validate=False)),
    ('imputer2', SimpleImputer(strategy='median')),
    (
        'quantile',
        QuantileTransformer(
            n_quantiles=50,
            output_distribution='normal',
            random_state=80,
        ),
    ),
])
seasonality_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('season', FunctionTransformer(month_seasonality, validate=False)),
])
final_categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])

final_preprocessor = ColumnTransformer([
    ('num_base', numeric_base_pipe, ['MONTH', 'TOTAL.CUSTOMERS']),
    ('log_customers', log_customers_pipe, ['TOTAL.CUSTOMERS']),
    ('quantile_anomaly', quantile_anomaly_pipe, ['ANOMALY.LEVEL']),
    ('seasonality', seasonality_pipe, ['MONTH']),
    ('cat', final_categorical_pipe, final_categorical),
])

final_model_pipe = Pipeline([
    ('preprocess', final_preprocessor),
    ('reg', RandomForestRegressor(random_state=80, n_jobs=-1)),
])

param_grid = {
    'reg__max_depth': [5, 10, None],
    'reg__min_samples_leaf': [5, 10, 25],
    'reg__n_estimators': [100, 200],
}

# 5-fold CV picks tree depth, leaf size, and number of estimators by minimizing RMSE.
grid_search = GridSearchCV(
    final_model_pipe,
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
)
grid_search.fit(X_train_final, y_train_final)

# Evaluate the grid-search winner on the held-out test set.
final_model = grid_search.best_estimator_
final_model.fit(X_train_final, y_train_final)
y_pred_final = final_model.predict(X_test_final)

final_rmse = root_mean_squared_error(y_test_final, y_pred_final)
final_r2 = r2_score(y_test_final, y_pred_final)

print('=== FINAL MODEL (RandomForestRegressor Pipeline) ===')
print('Best hyperparameters from GridSearchCV:', grid_search.best_params_)
print(f'CV RMSE (mean across folds): {-grid_search.best_score_:.2f} minutes')
print(f'Test RMSE: {final_rmse:.2f} minutes')
print(f'Test R^2 score: {final_r2:.4f}')
print('\nComparison to Step 6 baseline on the same prediction task:')
print(f'Baseline Test RMSE: {baseline_rmse:.2f} minutes')
print(f'Final Test RMSE:   {final_rmse:.2f} minutes')
print(f'Improvement: {baseline_rmse - final_rmse:.2f} minutes lower RMSE')

# Predicted vs actual scatter; dashed line is perfect prediction.
import plotly.graph_objects as go

fig_res = px.scatter(
    x=y_test_final,
    y=y_pred_final,
    labels={'x': 'Actual Duration (minutes)', 'y': 'Predicted Duration (minutes)'},
    title='Final Model: Predicted vs Actual Outage Duration',
)
max_val = max(y_test_final.max(), y_pred_final.max())
fig_res.add_trace(
    go.Scatter(
        x=[0, max_val],
        y=[0, max_val],
        mode='lines',
        name='Perfect prediction',
        line=dict(dash='dash', color='gray'),
    )
)
fig_res.update_layout(
    xaxis_title='Actual Outage Duration (minutes)',
    yaxis_title='Predicted Outage Duration (minutes)',
    legend_title='Reference',
    plot_bgcolor='white',
    title_font_size=16,
)
fig_res.show()
Path('assets').mkdir(exist_ok=True)
fig_res.write_html('assets/final_model_residuals.html', include_plotlyjs='cdn')


=== FINAL MODEL (RandomForestRegressor Pipeline) ===
Best hyperparameters from GridSearchCV: {'reg__max_depth': None, 'reg__min_samples_leaf': 10, 'reg__n_estimators': 100}
CV RMSE (mean across folds): 5482.67 minutes
Test RMSE: 4288.72 minutes
Test R^2 score: 0.3178

Comparison to Step 6 baseline on the same prediction task:
Baseline Test RMSE: 5179.97 minutes
Final Test RMSE:   4288.72 minutes
Improvement: 891.24 minutes lower RMSE


## Step 8: Fairness Analysis

We ask whether the **final duration model** predicts equally well for outages in high-population grid areas versus lower-population areas.

**Group definitions:**
- **Group X (high-impact):** Outages where `TOTAL.CUSTOMERS` is at or above the training-set median.
- **Group Y (low-impact):** Outages where `TOTAL.CUSTOMERS` is below the training-set median.

**Evaluation metric:** RMSE on the held-out test set, computed separately for each group.

**Hypotheses:**
- **Null Hypothesis (H₀):** The model is fair. RMSE for high-impact and low-impact outages are roughly the same; any observed difference is due to chance.
- **Alternative Hypothesis (Hₐ):** The model is unfair. RMSE for high-impact outages is **greater** than RMSE for low-impact outages (worse predictions where more people are served).

**Test statistic:** RMSE_high − RMSE_low.

**Significance level:** α = 0.05.

**Method:** Permutation test (2,000 repetitions) shuffling group labels on the test set while keeping Step 7 model predictions fixed—**no retraining** during the fairness test.

In [9]:
# Step 8: Fairness check—does RMSE differ between high- vs low-customer-impact test outages?
# Uses Step 7 predictions only (no retraining). Groups are split at the training median TOTAL.CUSTOMERS.

from sklearn.metrics import root_mean_squared_error

if 'y_pred_final' not in globals():
    raise RuntimeError('Run the Step 7 cell first so y_pred_final and related variables exist.')

rng_fair = np.random.default_rng(80)
reps = 2000
alpha = 0.05

fairness_df = pd.DataFrame({
    'y_true': y_test_final.to_numpy(),
    'y_pred': y_pred_final,
    'TOTAL.CUSTOMERS': X_test_final['TOTAL.CUSTOMERS'].to_numpy(),
})

cust_median = X_train_final['TOTAL.CUSTOMERS'].median()
fairness_df['impact_group'] = np.where(
    fairness_df['TOTAL.CUSTOMERS'] >= cust_median,
    'high_impact',
    'low_impact',
)


def rmse_for_group(frame, group_label):
    sub = frame[frame['impact_group'] == group_label]
    return root_mean_squared_error(sub['y_true'], sub['y_pred'])


rmse_high = rmse_for_group(fairness_df, 'high_impact')
rmse_low = rmse_for_group(fairness_df, 'low_impact')
observed_stat = rmse_high - rmse_low

print('Fairness groups defined by training-set median TOTAL.CUSTOMERS =', cust_median)
print(f'n(high_impact) = {(fairness_df["impact_group"] == "high_impact").sum()}')
print(f'n(low_impact)  = {(fairness_df["impact_group"] == "low_impact").sum()}')
print(f'RMSE (high_impact): {rmse_high:.2f} minutes')
print(f'RMSE (low_impact):  {rmse_low:.2f} minutes')
print(f'Observed test statistic (RMSE_high - RMSE_low): {observed_stat:.2f}')

groups = fairness_df['impact_group'].to_numpy()
y_true_arr = fairness_df['y_true'].to_numpy()
y_pred_arr = fairness_df['y_pred'].to_numpy()

simulated_stats = np.empty(reps)
for i in range(reps):
    perm_groups = rng_fair.permutation(groups)
    rmse_hi = root_mean_squared_error(
        y_true_arr[perm_groups == 'high_impact'],
        y_pred_arr[perm_groups == 'high_impact'],
    )
    rmse_lo = root_mean_squared_error(
        y_true_arr[perm_groups == 'low_impact'],
        y_pred_arr[perm_groups == 'low_impact'],
    )
    simulated_stats[i] = rmse_hi - rmse_lo

p_value = (np.sum(simulated_stats >= observed_stat) + 1) / (reps + 1)

print(f'\nPermutation test ({reps} repetitions):')
print(f'p-value: {p_value:.4f}')

if p_value < alpha:
    print(
        'Conclusion: Reject H0 at alpha = 0.05. '
        'Evidence that RMSE is higher for high-impact outages than for low-impact outages.'
    )
else:
    print(
        'Conclusion: Fail to reject H0 at alpha = 0.05. '
        'No statistically significant evidence that RMSE differs between impact groups.'
    )

fig_fair = px.histogram(
    simulated_stats,
    nbins=40,
    title='Fairness Permutation Null: RMSE(high-impact) − RMSE(low-impact)',
    labels={'value': 'RMSE difference (minutes)', 'count': 'Frequency'},
)
fig_fair.add_vline(
    x=observed_stat,
    line_dash='dash',
    line_color='red',
    annotation_text='Observed statistic',
)
fig_fair.update_layout(
    xaxis_title='RMSE(high-impact) − RMSE(low-impact) (minutes)',
    yaxis_title='Frequency',
    plot_bgcolor='white',
    title_font_size=16,
)
Path('assets').mkdir(exist_ok=True)
fig_fair.write_html('assets/fairness_permutation_null.html', include_plotlyjs='cdn')
fig_fair.show()


Fairness groups defined by training-set median TOTAL.CUSTOMERS = 3957980.0
n(high_impact) = 145
n(low_impact)  = 151
RMSE (high_impact): 5230.23 minutes
RMSE (low_impact):  3128.44 minutes
Observed test statistic (RMSE_high - RMSE_low): 2101.78

Permutation test (2000 repetitions):
p-value: 0.1179
Conclusion: Fail to reject H0 at alpha = 0.05. No statistically significant evidence that RMSE differs between impact groups.
